In [150]:
# 导入数据库

import pandas as pd
import pickle
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectFromModel, SelectKBest, RFE
from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC, SVC, SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

from sklearn.metrics import mean_squared_error, accuracy_score

In [118]:
# 导入数据

train_df = pd.read_csv('./training.csv', encoding='UTF-8')
print('null: \n', train_df.isnull().sum())
print('dup: \n', train_df[train_df.duplicated(['video_id', 'description_en'], keep=False)])
print('shape: \n', train_df.shape)
print('imb: \n', train_df['Rank'].value_counts(normalize=True))
train_df.head()

null: 
 user_id           0
video_id          0
name_en           0
description_en    0
Rank              0
dtype: int64
dup: 
         user_id  video_id     name_en  \
0         98234        13      Alive!   
1         19357        13      Alive!   
2        151726        13      Alive!   
3         69657        13      Alive!   
4          7935        13      Alive!   
...         ...       ...         ...   
376348   177946      1246  The String   
376349    90282      1246  The String   
376350    77769      1246  The String   
376351   202164      1246  The String   
376352    57044      1246  The String   

                                           description_en  Rank  
0       In this compelling documentary, five HIV-posit...     1  
1       In this compelling documentary, five HIV-posit...     1  
2       In this compelling documentary, five HIV-posit...     1  
3       In this compelling documentary, five HIV-posit...     1  
4       In this compelling documentary, five HIV-

,user_id,video_id,name_en,description_en,Rank
0,98234,13,Alive!,"In this compelling documentary, five HIV-posit...",1
1,19357,13,Alive!,"In this compelling documentary, five HIV-posit...",1
2,151726,13,Alive!,"In this compelling documentary, five HIV-posit...",1
3,69657,13,Alive!,"In this compelling documentary, five HIV-posit...",1
4,7935,13,Alive!,"In this compelling documentary, five HIV-posit...",1


In [128]:
# 数据清洗

# 删除重复的
train_df.drop_duplicates(['description_en', 'video_id'], keep='first', inplace=True)


print('dup: \n', train_df[train_df.duplicated(['user_id'], keep=False)])
print('shape: \n', train_df.shape)
print('imb: \n', train_df['Rank'].value_counts(normalize=True))

dup: 
         user_id  video_id                  name_en  \
0         98234        13                   Alive!   
12        19476        14     Appropriate Behavior   
2916       1544        26           The Commitment   
3558     204888        28                  Misfits   
3566     148214        29             Naz & Maalik   
...         ...       ...                      ...   
369536       43      1210          Don't Say Sorry   
374167    57662      1232  Woke Season 2 Episode 7   
374219    24369      1233  Woke Season 2 Episode 8   
374348    41011      1236             Grande École   
374724   127068      1238                 Being Me   

                                           description_en  Rank  \
0       In this compelling documentary, five HIV-posit...     1   
12      Shirin is struggling to become an ideal Persia...     2   
2916    Life is always dramatic with Sherwin and Paolo...     1   
3558    Tulsa, Oklahoma is a city in the midst of the ...     1   
3566    N

In [234]:
# 分词 -> 词形还原 -> 去停用词

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return None
    
    
def preprocessing_text(x, stop_words):
    arr = word_tokenize(x)
    tag_arr = pos_tag(arr)
    wnl = WordNetLemmatizer()
    lemm_arr = []
    for tag in tag_arr:
        wordnet_pos = get_wordnet_pos(tag[1]) or wordnet.NOUN
        lemm_arr.append(wnl.lemmatize(tag[0], wordnet_pos))
    return ' '.join([i for i in lemm_arr if i not in stop_words])


with open('./nlp/stop_words_en.txt', 'r', encoding='UTF-8') as f:
    stop_words = f.read().split('\n')

train_df['text'] = train_df['name_en'] + train_df['description_en']
train_df['text'] = train_df['text'].apply(lambda x: preprocessing_text(x, stop_words))
train_df.head()

,user_id,video_id,name_en,description_en,Rank,text
0,98234,13,Alive!,"In this compelling documentary, five HIV-posit...",1,Alive In compelling documentary five HIV-posit...
12,19476,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,2,Appropriate BehaviorShirin struggle become ide...
85,40124,15,Beautiful Something,"One night under a full moon in Philadelphia, t...",1,Beautiful SomethingOne night full moon Philade...
1035,96247,16,Boulevard,Nolan Mack is married. He has a respectable jo...,1,BoulevardNolan Mack married respectable job li...
1065,189210,17,Boy Meets Girl,This is a coming of age comedy about three twe...,5,Boy Meets GirlThis come age comedy three twent...


In [238]:
# 非结构数据转换为结构数据 -> 词袋法

vec_num = 0
vec_methods = [TfidfVectorizer(norm='l2', use_idf=True, sublinear_tf=True), CountVectorizer(), CountVectorizer(binary=True)]
vec_methods[vec_num].fit(train_df['text'])
word_mtx = vec_methods[vec_num].transform(train_df['text'])
word_mtx.shape

(1004, 8510)

In [239]:
# 非结构特征转换为结构特征 --词嵌入法

# model = Word2Vec.load('./nlp/word2vec_model.model')
# model.build_vocab(train_df['text'].apply(lambda x: x.split(' ')), update=True)
# model.train(train_df['text'].apply(lambda x: x.split(' ')), total_examples=model.corpus_count, epochs=model.epochs)
# model.save('./nlp/word2vec_model.model')

In [240]:
# 非结构特征转换为结构特征 --词嵌入法

# def word2sen(x, model):
#     arr = x.split(' ')
#     return sum([model.wv[i] for i in arr]) / len(arr)


# model = Word2Vec.load('./nlp/word2vec_model.model')
# train_df['text'] = train_df['text'].apply(lambda x: word2sen(x, model))
# word_mtx = pd.DataFrame(list(train_df['text']))
# word_mtx.shape

In [241]:
# 特征选择

de_num = 1
de_methods = [SelectFromModel(estimator=SVR()), SelectKBest(k=6300), RFE(estimator=LogisticRegression(), n_features_to_select=200)]
de_methods[de_num].fit(word_mtx, train_df['Rank'])
word_mtx_de = de_methods[de_num].transform(word_mtx)
word_mtx_de.shape

(1004, 6300)

In [242]:
# 数据划分

X_train, X_test, Y_train, Y_test = train_test_split(word_mtx_de, train_df['Rank'], test_size=0.2)
print('X_train: ', X_train.shape)
print('X_test: ', X_test.shape)
print('Y_train: ', Y_train.value_counts(normalize=True))
print('Y_test: ', Y_test.value_counts(normalize=True))

X_train:  (803, 6300)
X_test:  (201, 6300)
Y_train:  5    0.402242
1    0.336239
2    0.107098
3    0.083437
4    0.070984
Name: Rank, dtype: float64
Y_test:  5    0.412935
1    0.383085
2    0.114428
4    0.049751
3    0.039801
Name: Rank, dtype: float64


In [252]:
# 模型训练 -svm

svm_clf = SVR().fit(word_mtx_de, train_df['Rank'])
Y_pred = svm_clf.predict(X_test)
print(mean_squared_error(Y_test, Y_pred))

0.6339398082380294


In [244]:
# 模型训练 -lr

lr_clf = LogisticRegression().fit(X_train, Y_train)
Y_pred = lr_clf.predict(X_test)
print(mean_squared_error(Y_test, Y_pred))

5.159203980099503


In [245]:
# 模型训练 -naive_bayes

bayes_clf = BernoulliNB().fit(X_train, Y_train)
Y_pred = bayes_clf.predict(X_test)
print(mean_squared_error(Y_test, Y_pred))

5.666666666666667


In [253]:
# 保存模型

with open('./vec_method.pickle', 'wb') as f:
    pickle.dump(vec_methods[vec_num], f)
with open('./de_method.pickle', 'wb') as f:
    pickle.dump(de_methods[de_num], f)
with open('./clf.pickle', 'wb') as f:
    pickle.dump(svm_clf, f)